# The history matching workflow

History matching adjusts uncertain model parameters so that the model's predictions agree with observed data. In APEX you describe the problem in a TOML configuration file, validate it, run it, and read the posterior summary.

**Goal.** Run the same problem with two methods, ES-MDA and GIES, read what each run reports, and learn the terminal commands that inspect, stop and continue runs.

- **ES-MDA** keeps an *ensemble*, a set of candidate models. It updates the whole ensemble a fixed number of times against the same data. The data uncertainty is inflated at each pass, so every pass takes a gentler step.
- **GIES** is an iterative ensemble method. In each iteration it *tries* an update with the current damping. If the data mismatch improves, the trial is *accepted* and the damping is reduced a little. If not, the trial is *rejected*, the damping is raised, and it tries again. GIES stops when one of its stopping criteria is met (listed in Step 6) or the iteration cap is reached.

In a configuration, `[method.strategy]` picks the *update rule* used inside each analysis step (ES-MDA here uses `approx`, GIES uses `rlmmac`). `[method.schedule]` is a separate table that configures the *damping*. For GIES the default damping scheme is also called `rlmmac`, but it is a different thing from the update rule of the same name.

| Step | What |
|---|---|
| 1 | Copy a ready-made example from the examples catalogue |
| 2 | Read its ES-MDA configuration |
| 3 | Validate it and read the plan |
| 4 | Run it and wait for it |
| 5 | Read the posterior metrics |
| 6 | Repeat with GIES on the same problem and compare the two |
| 7 | Find the run directories |
| 8 | Inspect the runs from the terminal, then stop and continue a short run |

The example, *Linear-Gaussian: an exact posterior*, has a linear model with a Gaussian prior and Gaussian data. It is self-contained, so no simulator knowledge is needed. It is *synthetic*: the data were generated from a known true model, which lets APEX score the result against that truth.

**Prerequisites**

- APEX is installed. Run this notebook from the APEX folder (the folder where you type `uv run apex ...`). It creates the folders `work` and `apex_home` there, and the terminal commands further down use paths relative to the same folder.
- You know roughly what a prior, an observation and a posterior are.

## Step 1: copy an example to work on

The examples catalogue holds ready-made problems. The first cell below lists the ids in the catalogue (the web UI's Examples page shows the same catalogue). The second cell looks up `linear_gaussian` by its id and copies it into a `work` folder, where you can edit your copy freely.

The example holds two configurations for the same problem: `config_esmda.toml` (method ES-MDA) and `config_gies.toml` (method GIES).

In [1]:
from pathlib import Path

from apex.client import Client
from apex.services.examples import copy_example, examples_root, find_examples
from apex.services.settings import ExamplesSection

# The examples catalogue (the web UI's Examples page): copy one to work on.
catalogue = {e.id: e for e in find_examples(examples_root(ExamplesSection()))}
print("examples in the catalogue:", sorted(catalogue))

examples in the catalogue: ['3dbox_production', '3dbox_production_avo', 'linear_gaussian', 'optimization', 'optimization/3well', 'toy/multi_objective/bnh', 'toy/multi_objective/dtlz1', 'toy/multi_objective/dtlz2', 'toy/multi_objective/dtlz2_5', 'toy/multi_objective/fon', 'toy/multi_objective/kur', 'toy/multi_objective/sch', 'toy/multi_objective/srn', 'toy/multi_objective/tnk', 'toy/multi_objective/zdt1', 'toy/multi_objective/zdt2', 'toy/multi_objective/zdt3', 'toy/rastrigin', 'toy/toy_reservoir_npv_co2', 'toy/toy_reservoir_npv_water', 'toy/toy_reservoir_robust_npv']


In [2]:
copied = copy_example(catalogue["linear_gaussian"], Path("work"))
config = copied.folder / "config_esmda.toml"
print(config.relative_to(Path.cwd()))

work/linear_gaussian/config_esmda.toml


## Step 2: read the configuration

A run is fully described by its configuration file, so read it before you run it. The next cell prints the file without comments and blank lines. The sections mean:

- `[run]`: `name` labels the run, and `seed` fixes the random numbers so that the run can be repeated.
- `[[state.variables]]`: the unknowns we estimate. Here one variable `x` with three values.
- `[prior]`: the starting ensemble. `ensemble_size` is the number of members, and `generator` draws them, here independent normals (`gaussian`) with a `mean` and a `std`.
- `[observations]` and `[case]`: the data to match, and the true model (`truth_file`) used for scoring because this case is `synthetic`.
- `[simulator]`: the forward model. The `linear` simulator computes `y = G x` for a fixed matrix `G` (read from `forward_file`).
- `[execution]`: how simulations are run. `workers` sets how many run at the same time. It changes speed, not the problem.
- `[method]`: the algorithm and its options. The `inflation` list of ES-MDA has one entry per step, and each step is one *analysis step*: one update of the whole ensemble against the data. `strategy` names the update rule used inside each step. Here it is `approx`, with an `energy` option.

`energy` is an advanced option; `1.0` is a safe setting and you can leave it. It is a number in (0, 1] that says how much of the information in the *data anomalies* the update keeps. The data anomalies are how far each member's simulated data lies from the ensemble's average simulated data. With `1.0` the update uses all of it; lower values drop the weakest directions, which can steady the update when the data are noisy.

When you want to experiment, edit a copy of the configuration, give it another `[run]` name, and validate it before running it.

In [3]:
def setting(line):
    for index, char in enumerate(line):
        if char == "#" and line[:index].count('"') % 2 == 0:
            return line[:index].rstrip()
    return line.rstrip()


settings = [setting(line) for line in config.read_text().splitlines()]
print("\n".join(line for line in settings if line))

[run]
name = "linear_esmda"
seed = 2026
[[state.variables]]
name = "x"
size = 3
[prior]
ensemble_size = 500
generator = { name = "gaussian", options = { mean = 0.0, std = 1.0 } }
[observations]
file = "observations.csv"
[case]
kind = "synthetic"
truth_file = "truth.npz"
[simulator]
name = "linear"
[simulator.options]
forward_file = "forward.npz"
variables = ["x"]
outputs = [["t0", "y"], ["t1", "y"], ["t2", "y"], ["t3", "y"], ["t4", "y"], ["t5", "y"], ["t6", "y"], ["t7", "y"]]
[execution]
name = "process"
options = { workers = 4 }
[method]
name = "esmda"
options = { inflation = [4, 4, 4, 4] }
strategy = { name = "approx", options = { energy = 1.0 } }


## Step 3: validate and look at the plan

`Client(Path("apex_home"))` is a client on a services root of this notebook's own: the folder `apex_home`. It keeps the records and logs of what you submit through this client, so they stay apart from any other work.

`validate` checks the configuration and plans its cost, and nothing is simulated. The plan reports the method, the number of members and the most simulator runs the configuration could use. For an expensive simulator that maximum is the most you could pay, so read it before you start a run.

For ES-MDA the count is easy to check by hand. Four inflation entries give four analysis steps, so there are 5 ensembles: the prior (iteration 0) plus 4 updated ones. Each ensemble has one run per member, so 5 × 500 = 2500, which is what the plan should show.

In [4]:
client = Client(Path("apex_home"))  # a services root of its own
checked = client.validate(config)
print("valid:", checked.valid)
plan = checked.plan
print(f"{plan.method}: {plan.members} members, at most {plan.max_simulator_runs} simulator runs")

valid: True
esmda: 500 members, at most 2500 simulator runs


## Step 4: run and wait

`submit_run` submits the configuration and gives back a handle. `wait` takes the handle's id and waits for the run to end, here for at most `timeout_s` seconds, polling every `poll_s` seconds. It returns a status whose `state` we print.

Read the state before you trust any summary: only `complete` is a finished run. If you see anything else, validate and read your configuration again, or raise `timeout_s` if the wait ran out.

In [5]:
handle = client.submit_run(config)
status = client.wait(handle.id, timeout_s=600, poll_s=0.5)
print(status.state)

complete


## Step 5: read the posterior summary

A submission has its own id, `handle.id`, and holds one or more runs, each with its own run id in `handle.run_ids`. Waiting uses the submission id. The summary belongs to a run, so it uses a run id. This example holds a single run, so we take index 0.

The iterations start at 0, the prior ensemble before any update. The last iteration is the posterior. With four inflation entries, expect `[0, 1, 2, 3, 4]`. The metrics are reported per variable and as `all`:

- `normalized_data_mismatch`: the mean of `((d_obs - y) / sigma)^2` over the data, the squared distance between observed and simulated data in units of the observation uncertainty `sigma`. Lower is a closer fit. Dividing by `sigma` puts data of different size and uncertainty on one scale, so use this one to judge and compare fit.
- `data_mismatch`: also printed. Its scale depends on the units and uncertainties of the data, so do not compare it with the normalized values.
- `rmse`: the root-mean-square error of the parameters against the true model. It needs a known truth, which this synthetic case has. Lower is better.
- `model_change`: the change of each member's parameters since the ensemble it was updated from, so a rough size of the last step. It is not the distance from the prior.

In [6]:
summary = client.run_summary(handle.run_ids[0])
print(summary.method, summary.state, "iterations:", summary.iterations)
for name, values in summary.posterior_metrics.items():
    print(name, values)

esmda complete iterations: [0, 1, 2, 3, 4]
data_mismatch {'all': 18.92859148916899, 'y': 18.92859148916899}
normalized_data_mismatch {'all': 1.386665921591923, 'y': 1.386665921591923}
rmse {'x': 0.20097220595156404, 'all': 0.20097220595156404}
model_change {'x': 0.1040374288027935, 'all': 0.1040374288027935}


## Step 6: the same problem with GIES

The example also holds `config_gies.toml`: the same problem with method GIES. The next cell prints its `[method]` section and its plan. The section differs from ES-MDA:

- `strategy` selects the update rule (`rlmmac`).
- `schedule` configures the damping, which controls the size of the update steps: more damping means smaller, more cautious steps. Options you can set:
  - `initial`: the damping of the first analysis; it must be positive.
  - `factor`: what the damping is multiplied by after a rejected trial (greater than 1, default 2).
  - `reduction`: what the damping is multiplied by after an accepted trial (default 0.9).
  - `data_mismatch_tolerance`: when an improving trial changes the mean data mismatch by a relative amount below this (default 0.01), the run has converged and **stops**. A smaller value lets the run continue until the improvements are tiny.
  - `max_stalled_rejections` (default 3): a rejected trial is *stalled* when it changed the mean data mismatch by a relative amount below `data_mismatch_tolerance`, so it was not better and barely different. The run stops after this many **consecutive** stalled rejections. Any other trial resets the count to zero. A rejection that is not stalled still raises the damping, and the next trial retries with it.
- `max_iterations` caps the number of iterations. The run ends at the cap if no other criterion stopped it first.

An example with a tolerance of 0.01 and `max_stalled_rejections = 3`:

| Trial | Outcome | Relative change of the mismatch | Stalled count |
|---|---|---|---|
| 1 | accepted | 12% better | 0 |
| 2 | rejected | 30% worse | 0 (not stalled, the change is large) |
| 3 | rejected | 0.4% worse | 1 |
| 4 | rejected | 0.2% worse | 2 |
| 5 | rejected | 0.1% worse | 3, so the run stops |

If trial 4 had been accepted instead, the count would return to 0.

So a GIES run ends when the tolerance is met, when rejections stall, or at the cap, whichever comes first. Always read the values in your own file, not the defaults.

GIES has no fixed iteration list, so the plan is a ceiling for the worst case, and a run may stop much earlier. It also allows for extra trials, so it need not equal a simple count of ensembles times members; read it from the plan that `validate` prints.

In [7]:
gies_config = copied.folder / "config_gies.toml"
method_lines = []
in_method = False
for line in gies_config.read_text().splitlines():
    text = setting(line)
    if text.strip().startswith("["):
        in_method = text.strip() == "[method]"
    if in_method and text.strip():
        method_lines.append(text)
print("\n".join(method_lines))

gies_checked = client.validate(gies_config)
print("valid:", gies_checked.valid)
gies_plan = gies_checked.plan
print(
    f"{gies_plan.method}: {gies_plan.members} members, "
    f"at most {gies_plan.max_simulator_runs} simulator runs"
)

[method]
name = "gies"
strategy = { name = "rlmmac", options = { energy = 1.0 } }
schedule = { initial = 1.0, factor = 2.0, data_mismatch_tolerance = 1e-4 }
max_iterations = 20
valid: True
gies: 500 members, at most 10541 simulator runs


Now run it, and estimate how much of the planned ceiling it used. Every ensemble simulated, the prior (iteration 0) included, costs one run per member. So the number of ensembles times the ensemble size is the number of simulator runs when every iteration simulates one ensemble. If some iterations needed extra trials, the real count is higher, so treat the product as a lower bound.

In [8]:
gies_handle = client.submit_run(gies_config)
gies_status = client.wait(gies_handle.id, timeout_s=600, poll_s=0.5)
print(gies_status.state)

gies_summary = client.run_summary(gies_handle.run_ids[0])
print(gies_summary.method, gies_summary.state, "iterations:", gies_summary.iterations)
for name, values in gies_summary.posterior_metrics.items():
    print(name, values)

ensembles = len(gies_summary.iterations)
used = ensembles * gies_plan.members
print(f"ensembles simulated (including iteration 0): {ensembles}")
print(
    f"{ensembles} x {gies_plan.members} = {used} simulator runs at least, "
    f"against a planned maximum of {gies_plan.max_simulator_runs}"
)
print(f"fraction of the ceiling used (at least): {used / gies_plan.max_simulator_runs:.0%}")

complete
gies complete iterations: [0, 1, 2, 3, 4, 5, 6, 7, 8]
data_mismatch {'all': 13.129146714443902, 'y': 13.129146714443902}
normalized_data_mismatch {'all': 1.4292791906552078, 'y': 1.4292791906552078}
rmse {'x': 0.20702499399688715, 'all': 0.20702499399688715}
model_change {'x': 0.002343765845202029, 'all': 0.002343765845202029}
ensembles simulated (including iteration 0): 9
9 x 500 = 4500 simulator runs at least, against a planned maximum of 10541
fraction of the ceiling used (at least): 43%


The ceiling is a worst case; a run that converges early pays only for the ensembles it simulated.

## Comparing the two runs

Both runs start from the same prior and data, with the same ensemble size and seed, so metrics with the same name can be put side by side. That does not make the runs equal in cost: they use different numbers of iterations and simulator runs. The table shows the `all` entry of each metric.

In [9]:
import pandas as pd

rows = {}
for name, values in summary.posterior_metrics.items():
    other = gies_summary.posterior_metrics.get(name, {})
    rows[name] = {"ES-MDA": values.get("all"), "GIES": other.get("all")}
table = pd.DataFrame(rows).T
print(table.to_string())

                             ES-MDA       GIES
data_mismatch             18.928591  13.129147
normalized_data_mismatch   1.386666   1.429279
rmse                       0.200972   0.207025
model_change               0.104037   0.002344


### Reading the table

The next cell reads the table metric by metric, saying which run is lower and by how much. Lower is better for the mismatch and `rmse` rows. `model_change` is only the size of the last step, so it is not ranked.

In [10]:
for name in table.index:
    if name == "model_change":
        continue
    a = table.loc[name, "ES-MDA"]
    g = table.loc[name, "GIES"]
    lower = "GIES" if g < a else "ES-MDA"
    gap = abs(g - a) / max(a, g) * 100
    print(f"{name}: lower for {lower}, by {gap:.1f}% of the larger value")

print(
    "model_change: ES-MDA",
    table.loc["model_change", "ES-MDA"],
    "GIES",
    table.loc["model_change", "GIES"],
)

data_mismatch: lower for GIES, by 30.6% of the larger value
normalized_data_mismatch: lower for ES-MDA, by 3.0% of the larger value
rmse: lower for ES-MDA, by 2.9% of the larger value
model_change: ES-MDA 0.1040374288027935 GIES 0.002343765845202029


How to read this output:

- **Mismatch rows**: the lower value fits the data more closely. The two mismatch measures are on different scales and can disagree on the ranking. Judge fit by `normalized_data_mismatch`, and when its gap between the runs is small, the runs differ little in fit and fit alone cannot pick a winner.
- **rmse**: how near each posterior is to the true model. We can read this only because the case is synthetic.
- **model_change**: a much smaller last step means a run had nearly stopped moving at its end. That says nothing about whether it is better or worse.

In a real case there is no truth, so compare methods through the fit, the iteration-by-iteration mismatch history from `apex report` (Step 8), the size of `model_change`, and above all the quality of the forecasts the posteriors give.

## Step 7: find the run directories

Each run has a directory of its own: its configuration snapshot, history and summary, and a `checkpoints` folder. A *checkpoint* is a saved copy of the run's state after an analysis step, from which a stopped run can continue (Step 8). The run summaries name the directory in `path`. For the runs of this notebook the directories are `work/linear_gaussian/runs/<name>`, where `<name>` is the `[run]` name: the run directory sits in a `runs` folder next to the configuration file.

In [11]:
def relative(path_text):
    path = Path(path_text)
    try:
        return path.relative_to(Path.cwd())
    except ValueError:
        return path


esmda_dir = relative(summary.path)
gies_dir = relative(gies_summary.path)
for name, run_dir in [("ES-MDA", esmda_dir), ("GIES", gies_dir)]:
    print(f"{name}: {run_dir}")
    print("   ", sorted(child.name for child in run_dir.iterdir()))

ES-MDA: work/linear_gaussian/runs/linear_esmda
    ['checkpoints', 'config.json', 'events.jsonl', 'history.json', 'provenance.json', 'run.lock', 'snapshot_version', 'state.json', 'summary.json']
GIES: work/linear_gaussian/runs/linear_gies
    ['checkpoints', 'config.json', 'events.jsonl', 'history.json', 'provenance.json', 'run.lock', 'snapshot_version', 'state.json', 'summary.json']


## Step 8: inspect from the terminal

The terminal commands below read the runs the cells made, so **run every cell above first**, then type the commands in order from the APEX folder (the folder you ran the notebook in). Their paths start at `work`.

### Summaries and reports of finished runs

`apex summary` takes run directories and lists every metric at every step: the iteration (0 for the prior, then each step) and the value of each metric at that iteration, for each run. Compare it with the Python output above: the last iteration of each run should show the same posterior metrics as Steps 5 and 6, and ES-MDA should list iterations 0 to 4.

```bash
uv run apex summary work/linear_gaussian/runs/linear_esmda work/linear_gaussian/runs/linear_gies
```

`apex report` takes one run directory and prints its data mismatch history (the mismatch at each iteration, in iteration order) and performance metrics. Look at the mismatch history first: it should fall from the prior (iteration 0) and then level off. A mismatch that keeps rising, or does not move at all, is a sign to revisit the configuration. ES-MDA takes a fixed number of steps, while GIES stops by its own criteria, so the histories differ in length.

```bash
uv run apex report work/linear_gaussian/runs/linear_esmda
uv run apex report work/linear_gaussian/runs/linear_gies
```

With `--plots` it also writes the result figures (PNG and PDF), and `--output` takes the folder to write them to (the default is a `report` folder inside the run directory). This writes the ES-MDA figures to a folder named `figs`, in the folder you run the command from; list it afterwards with `ls figs` and open the files. The one to look at first is the mismatch history: a good result is a curve that falls from the prior and then levels off.

```bash
uv run apex report work/linear_gaussian/runs/linear_esmda --plots --output figs
```

### Starting, stopping and continuing runs

This notebook started its runs from Python with `submit_run`. From the terminal the equivalent is `apex run CONFIG`, which takes a configuration file and starts a run. The run directory is `runs/<name>` next to the configuration file, with `<name>` taken from the `[run]` table's `name`, **not** from the file name.

`config_esmda.toml` and `config_gies.toml` were already run by the cells above, so never `apex run` them again: their run directories exist. To run something new from the terminal, use a copy with another `[run]` name. The next cell writes a copy of the ES-MDA configuration as `config_esmda_short.toml`, with the `[run]` name changed to `linear_esmda_short`. That name decides the run directory, `work/linear_gaussian/runs/linear_esmda_short`. The file sits next to the original so that its data files are found.

In [12]:
short = copied.folder / "config_esmda_short.toml"
text = config.read_text()
if 'name = "linear_esmda"' not in text:
    raise ValueError("the run name line was not found in the configuration")
short.write_text(text.replace('name = "linear_esmda"', 'name = "linear_esmda_short"', 1))
print(short.relative_to(Path.cwd()))

work/linear_gaussian/config_esmda_short.toml


`apex validate` takes a configuration and optionally a services root with `--root`. It prints the plan without simulating anything. Expect the method `esmda` and 500 members, with the same planned maximum you read in Step 3:

```bash
uv run apex validate work/linear_gaussian/config_esmda_short.toml --root apex_home
```

An *analysis step* is one update of the ensemble against the data. For ES-MDA it is the same thing as one iteration, so a configuration with four `inflation` entries has four analysis steps. A run that stops, on purpose or by interruption, continues from its last checkpoint (Step 7) with `apex resume` instead of starting over.

`--max-steps` on `apex run` is the predictable way to stop: the run ends after exactly that many analysis steps, saves its state, and can be continued. Here we run two of the four steps, look at the summary, then continue with `apex resume`, which takes the run directory. Without `--max-steps` the resumed run carries on until it ends. Then we look again. Run the four commands in this order; the two `apex summary` calls are the ones that print tables:

```bash
uv run apex run work/linear_gaussian/config_esmda_short.toml --max-steps 2
uv run apex summary work/linear_gaussian/runs/linear_esmda_short
uv run apex resume work/linear_gaussian/runs/linear_esmda_short
uv run apex summary work/linear_gaussian/runs/linear_esmda_short
```

How to check each stage:

- After the first command the run has stopped on purpose, after two analysis steps, and is not a finished run. The first `apex summary` should list only iterations 0, 1 and 2: the prior and the two steps taken. Do not take that table for a posterior.
- After `apex resume` the second `apex summary` should list iterations 0 to 4: the prior and all four steps. If the first summary already shows all five, the stop did not take effect; if a command prints an error, read it before you resume.

### Stopping a run that is still going

For a long run that you did not limit in advance, `apex stop` takes the run directory of a run that is still going and asks it to stop at its next checkpoint. The command is shown here but not as a step to run, because the runs in this notebook finish in seconds, faster than you could type it:

```text
uv run apex stop work/linear_gaussian/runs/<name>
```

You would type it in a second terminal while `apex run` is still busy in the first. The first terminal then returns once the run has saved its state, and `apex summary` lists only the iterations saved so far. `apex resume` continues from there. If you start a long run of your own, give `apex resume` a `--max-steps` too, so the continuation also ends when you expect it to.

## Pitfalls

- **Skipping validation.** Validate first. It catches configuration mistakes before any simulator time is spent.
- **Ignoring the plan.** The maximum number of simulator runs is the most you can pay, so read it before you start. For GIES it is a worst-case ceiling, and the run may use much less.
- **Reading a summary of a run that did not end in `complete`.** Print the state first. A run stopped with `--max-steps` shows only the iterations done so far.
- **Reading `rmse` for a real case.** It is scored against a known true model, which only a synthetic case has.
- **Choosing a method from one metric.** Read the mismatch together with the mismatch history and, for real cases, the forecasts. The two mismatch measures can rank runs differently; rely on the normalized one.
- **Confusing strategy and schedule.** `strategy` is the update rule; `schedule` is the damping setting. Check the values written in your configuration, not the defaults.
- **Confusing the file name with the run name.** The run directory is named after the `[run]` name inside the file. Take the path from the run summary (`path`).
- **Running the terminal commands from another folder.** The paths above start at `work`, so run them from the folder where the notebook ran.
- **Running a configuration that already ran.** Its run directory exists. Resume it with `apex resume`, or run a copy with another `[run]` name.

## What to read next

- Change `ensemble_size` or the `inflation` list in a copy of the configuration, validate, and run it under another `[run]` name to see how the plan and the metrics change.
- Open the figures that `apex report --plots --output figs` wrote to the `figs` folder, starting with the mismatch history.
- Browse the examples catalogue for problems closer to your own: list the ids with `find_examples` as in Step 1, or open the web UI's Examples page.
- Forecasting beyond history: `apex predict` takes a prediction configuration, a TOML file of its own that you write for a finished history-matching run, and forecasts that run's ensembles beyond the history. List its options with:

```bash
uv run apex predict --help
```